# 🛡️ Mark3: 3-Class Communications Threat Classification System
### (0: Ham, 1: Spam, 2: Phishing / Smishing)

---

## 📚 Literature-Backed Architecture Justification
The decision to isolate URL analysis and focus purely on semantic text classification across three categories is highly supported by recent cybersecurity literature:

1. **3-Class Taxonomy**: Recent studies validate structuring datasets into **ham, spam, and smishing/phishing** categories to accurately model the modern mobile and email threat landscape.
2. **Deep Learning (GRU/LSTM)**: Research demonstrates that sequential deep learning models, specifically utilizing **Gated Recurrent Units (GRUs)**, are highly effective at identifying the complex context and structure of phishing attacks in short messages.
3. **Semantic Text Focus**: Leveraging Natural Language Processing (NLP) to scrutinize linguistic patterns and contextual clues—independent of external URL reputation checks—is an established, effective mechanism for real-time threat detection.

---

## ⚙️ 4 Technical Pro-Tips Implemented
1. **`<URL>` Token Preservation**: URLs are replaced with `urltoken` so alphanumeric regex cleaning (`re.sub(r'[^a-z0-9\s]', '', text)`) preserves the tag intact rather than stripping `<` and `>`.
2. **Length Discrepancies (SMS vs Email)**: Fixed sequence length set to `maxlen=150` with `padding='post'` and `truncating='post'` to capture critical initial payload without overloading GRU memory.
3. **Class Imbalance**: Calculated balanced class weights using `sklearn.utils.class_weight.compute_class_weight` and passed into loss computation.
4. **LIME Compatibility**: Wrapped prediction in `predict_pipeline(texts)` accepting `list[str]` and returning a `(N, 3)` probability matrix.

## 1. Data Preparation & Feature Engineering

In [ ]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import kagglehub

from preprocessing import clean_text, standardize_labels, calculate_class_weights
from models import BaselineMLModel, GRUClassifier
from explainability import ThreatExplainer

print('Downloading Datasets via KaggleHub...', flush=True)
path1 = kagglehub.dataset_download('akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000')
path2 = kagglehub.dataset_download('dharshiyanacc/spam-ham-and-phishing-message-dataset-for-nlp')
print('Dataset paths:', path1, path2, flush=True)

In [ ]:
# Load and Standardize Datasets
def find_csvs(dir_path):
    res = []
    for root, _, files in os.walk(dir_path):
        for f in files:
            if f.endswith('.csv'):
                res.append(os.path.join(root, f))
    return res

dfs = []
for csv in find_csvs(path1) + find_csvs(path2):
    print(f'Ingesting {os.path.basename(csv)}...', flush=True)
    df_raw = pd.read_csv(csv)
    text_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ['text', 'message', 'email', 'body'])]
    label_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ['label', 'target', 'category', 'type', 'class'])]
    if text_cols and label_cols:
        std_df = standardize_labels(df_raw, text_cols[0], label_cols[0], max_samples=40000)
        dfs.append(std_df)

df = pd.concat(dfs, ignore_index=True).drop_duplicates()
print(f'Total Unified Samples: {len(df)}', flush=True)
print('\nClass Distribution:', flush=True)
print(df['target'].value_counts(), flush=True)

In [ ]:
# Stratified Train / Validation / Test Split
X = df['cleaned_text'].values
y = df['target'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# Compute Class Weights
class_weights = calculate_class_weights(y_train)
print('Class Weights for Training:', class_weights, flush=True)

## 2. Baseline Machine Learning Model (TF-IDF + Naive Bayes)

In [ ]:
baseline = BaselineMLModel(max_features=10000)
print('Training Naive Bayes Baseline...', flush=True)
baseline.train(X_train, y_train)

acc, report, cm = baseline.evaluate(X_test, y_test, target_names=['Ham', 'Spam', 'Phishing'])
print(f'Baseline Model Accuracy: {acc*100:.2f}%', flush=True)
print('\nClassification Report:\n', report, flush=True)

# Plot Seaborn Confusion Matrix Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam', 'Phishing'], yticklabels=['Ham', 'Spam', 'Phishing'])
plt.title('Baseline Naive Bayes Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 3. Lightweight Deep Learning Model (Embedding + GRU Network)

In [ ]:
# Initialize & Train Lightweight GRU Model
gru = GRUClassifier(vocab_size=10000, embed_dim=128, hidden_dim=64, num_classes=3, max_len=150)

train_sample_size = min(30000, len(X_train))
print(f'Training GRU on {train_sample_size} samples...', flush=True)
gru.train(
    X_train[:train_sample_size], y_train[:train_sample_size],
    X_val, y_val,
    class_weights=class_weights,
    epochs=5,
    batch_size=64,
    patience=2
)

# Plot Loss & Accuracy Curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(gru.history['train_loss'], label='Train Loss', marker='o')
ax[0].plot(gru.history['val_loss'], label='Validation Loss', marker='s')
ax[0].set_title('GRU Training vs Validation Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].legend()

ax[1].plot(gru.history['train_acc'], label='Train Accuracy', marker='o')
ax[1].plot(gru.history['val_acc'], label='Validation Accuracy', marker='s')
ax[1].set_title('GRU Training vs Validation Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
plt.show()

In [ ]:
# Extract discrete integer class labels (0, 1, 2) using np.argmax on probabilities
dl_preds_probs = gru.predict_proba(X_test)
dl_preds_labels = np.argmax(dl_preds_probs, axis=1)

dl_cm = confusion_matrix(y_test, dl_preds_labels)
print('GRU Classification Report:\n', classification_report(y_test, dl_preds_labels, target_names=['Ham', 'Spam', 'Phishing']), flush=True)

plt.figure(figsize=(7, 5))
sns.heatmap(dl_cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Ham', 'Spam', 'Phishing'], yticklabels=['Ham', 'Spam', 'Phishing'])
plt.title('Deep Learning GRU Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 4. Explainable AI (XAI) with LIME

In [ ]:
# Force reload updated modules in Jupyter Kernel memory
import importlib
import models
import explainability
importlib.reload(models)
importlib.reload(explainability)
from explainability import ThreatExplainer
from IPython.display import display, HTML

sample_phishing_msg = "URGENT ACTION REQUIRED: Your account security was compromised. Verify password now at http://account-update-sec.com/login"

explainer = ThreatExplainer(gru.predict_proba)
exp, pred_class = explainer.explain_instance(sample_phishing_msg, num_features=8)

print(f'Input Message: "{sample_phishing_msg}"', flush=True)
print(f'Model Prediction: {["Ham", "Spam", "Phishing"][pred_class]}', flush=True)

print('\nTop Feature Importances:', flush=True)
for token, score in exp.as_list():
    print(f"  Token: '{token:<15}' Importance Weight: {score:+.4f}", flush=True)

# Display LIME Text Explanation HTML
display(HTML(exp.as_html()))